# Zein – kappa-casein Protein-Protein Docking (MEGADOCK, GPU-accelerated)

This notebook is adapted from a MEGADOCK tutorial (Ohue et al., MEGADOCK 4.0, *Bioinformatics* 2014) to dock
**zein-α** (AlphaFold-predicted structure) against **kappa-casein** (64-residue fragment, `kappa_casein_frag.pdb`).

This is a single-pair notebook — zein vs each dairy protein has its own separate notebook, so each run
stays fast and its outputs don't get mixed together.

**Why switch from LightDock:** LightDock's GSO-based rigid-body+ANM search took 20+ hours locally per protein
pair on CPU. MEGADOCK uses an FFT-grid-based exhaustive rigid-body search accelerated on a GPU — the same
class of algorithm as ZDOCK/ClusPro/HDOCK. On a Colab GPU (free tier T4), a full docking run typically finishes
in minutes, not hours.

**Trade-off to be aware of:** MEGADOCK is rigid-body only (no ANM/backbone flexibility like LightDock had).
This isn't a step backward for zein/casein specifically — both are largely disordered proteins where ANM's
small (~0.5Å) deformations weren't capturing real flexibility anyway — but it does mean results should be
treated the same way as your earlier HDOCK/ClusPro runs: exploratory ranking, not definitive structures.

**Restraint handling:** this notebook uses MEGADOCK's `block` tool to mask out all zein receptor residues
*except* the 40 residues in your `zein_restricted.txt` restraint list, so the FFT search is steered toward
your known target patch instead of exploring the whole (mostly hydrophobic, non-interacting) zein surface.
This is a different mechanism than LightDock's restraint-biased swarms, but serves the same purpose.

**License note:** MEGADOCK is CC BY-NC 4.0 — free for academic/research use (this project), but commercial
use requires permission from Tokyo Institute of Technology.


## 1. Set up Colab runtime

Before running anything: **Runtime → Change runtime type → GPU** (T4 is fine, it's what this was tested on).

In [ ]:
# @title Install MEGADOCK and dependencies (GPU build)

# Clone MEGADOCK
!git clone https://github.com/akiyamalab/MEGADOCK

# Clone NVIDIA cuda-samples to provide helper_cuda.h
!git clone https://github.com/NVIDIA/cuda-samples.git /content/cuda-samples

# FFT library required by MEGADOCK
!apt-get install -y libfftw3-dev libfftw3-single3

# Build the GPU binary
%cd /content/MEGADOCK
!make -j 2 -f Makefile.colab

# Python deps for structure handling + visualization
!pip install -q biopython
!pip install -q nglview==3.0.8
!jupyter-nbextension enable nglview --py --sys-prefix

## 2. Connect Google Drive and locate your structures

This uses a Google Drive mount, so you only place the files once and every future run of this notebook can
read them directly without re-selecting files each time.

**One-time setup (do this before running the cell below), if you haven't already:**

1. Go to [Google Drive](https://drive.google.com) in your browser.
2. Create a folder named `zein_casein_docking` (or any name — just update `DRIVE_FOLDER` below to match).
3. Upload these 3 files into that folder directly, flat, no subfolders needed:
   - `zein_model.pdb`
   - `zein_restricted.txt`
   - `kappa_casein_frag.pdb` (lives locally at `protein_structures/casein/kappa_casein_frag.pdb`)

If your Drive folder already has the shared `zein_model.pdb` / `zein_restricted.txt` files in it from the other notebook, that's fine too — this notebook only additionally needs `kappa_casein_frag.pdb`.

If you have Google Drive for Desktop installed and it's already syncing your `UMD-work` folder, you don't
need to re-upload anything manually — just point `DRIVE_FOLDER` below at wherever that folder lives inside
your Drive instead.


In [ ]:
# @title Mount Google Drive and copy the structure files into the working directory
from google.colab import drive
drive.mount('/content/drive')

# Change this if you used a different folder name / location in your Drive
DRIVE_FOLDER = "/content/drive/MyDrive/UMD/Molecular_Docking/Megadocking/zein_casein_docking"

import os, shutil

required_files = ["zein_model.pdb", "zein_restricted.txt", "kappa_casein_frag.pdb"]
missing = [f for f in required_files if not os.path.exists(os.path.join(DRIVE_FOLDER, f))]

if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {DRIVE_FOLDER}. Upload them to that Drive folder (see instructions above), "
        f"or update DRIVE_FOLDER to point at the correct path, then re-run this cell."
    )

for fname in required_files:
    shutil.copy(os.path.join(DRIVE_FOLDER, fname), f"/content/MEGADOCK/{fname}")

print("Copied from Drive into /content/MEGADOCK:")
for fname in required_files:
    print(" -", fname)


## 3. Build the restraint-blocked zein receptor

MEGADOCK's `block` tool marks specified receptor residues as excluded from the binding-site search
(renamed to `BLK` internally). We block **everything except** the 40 residues in `zein_restricted.txt`,
so the FFT search is steered toward that patch.

In [ ]:
# @title Compute the blocked-residue list and apply blocking (pure Python 3)
#
# MEGADOCK's `block` tool ships as a Python 2 script, and modern Colab images
# don't have a Python 2 interpreter for its shebang to resolve -- that's what
# caused the "FileNotFoundError: ./block" error. Its logic is simple (rename
# the residue field to "BLK" for excluded residues), so it's reimplemented
# directly here instead of shelling out to the original script.

RECEPTOR_PDB = "zein_model.pdb"
RESTRAINTS_FILE = "zein_restricted.txt"
BLOCKED_RECEPTOR = "zein_blocked.pdb"

with open(RESTRAINTS_FILE) as f:
    line = f.read().strip()
keep_str, chain = line.split(":")
keep_residues = set(int(x) for x in keep_str.split(","))

# Find the full residue range actually present in the PDB for that chain
all_residues = set()
with open(RECEPTOR_PDB) as f:
    for l in f:
        if l.startswith("ATOM") and l[21].strip() == chain:
            all_residues.add(int(l[22:26]))

block_residues = all_residues - keep_residues

print(f"Chain: {chain}")
print(f"Total residues: {len(all_residues)}")
print(f"Keeping (restraint) residues: {len(keep_residues)}")
print(f"Blocking: {len(block_residues)} residues")

def block_line(l, chain, block_set):
    """Reimplementation of MEGADOCK's block script logic."""
    if not (l.startswith("ATOM") or l.startswith("HETATM")):
        return l
    if l[21] != chain:
        return l
    try:
        resnum = int(l[22:26])
    except ValueError:
        return l
    if resnum not in block_set:
        return l
    return l[0:16] + " BLK" + l[20:]

with open(RECEPTOR_PDB) as fin, open(BLOCKED_RECEPTOR, "w") as fout:
    for l in fin:
        fout.write(block_line(l.rstrip("\n"), chain, block_residues) + "\n")

print(f"Wrote {BLOCKED_RECEPTOR}")


## 4. Set docking parameters

`-t 3 -N 10800` matches the settings MEGADOCK's own docs recommend for `ppiscore`-compatible runs (see `doc/README.md`). Default rotational sampling (`-r 3600`, i.e. 15° steps) is used.

In [ ]:
# @title MEGADOCK parameters
T = 3        # predictions kept per rotation (megadock -t)
N = 10800    # total output predictions retained (megadock -N)

LIGAND_NAME = "kappa_casein"
LIGAND_PDB = "kappa_casein_frag.pdb"

print(f"Receptor (blocked): {BLOCKED_RECEPTOR}")
print(f"Ligand: {LIGAND_NAME} ({LIGAND_PDB})")
print(f"t={T}, N={N}")


## 5. Run MEGADOCK

This is the step that took 20+ hours locally with LightDock. On a Colab GPU this should take on the order of minutes (exact time depends on protein size and current GPU load) — watch the elapsed-time line MEGADOCK prints at the end of the run.

In [ ]:
# @title Run MEGADOCK: zein (blocked) vs. kappa-casein
import subprocess, time

outfile = f"dock_{LIGAND_NAME}.out"
print(f"\n=== Docking zein vs {LIGAND_NAME} ===")
t0 = time.time()
cmd = ["./megadock-gpu", "-R", BLOCKED_RECEPTOR, "-L", LIGAND_PDB,
       "-t", str(T), "-N", str(N), "-o", outfile]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:])  # tail of the log
if result.returncode != 0:
    print("STDERR:", result.stderr)
print(f"Elapsed (wall clock, incl. overhead): {time.time()-t0:.1f} sec")
print(f"\nDone. Output file: {outfile}")


## 6. PPI score

Quick sanity check — gives an overall interaction-likelihood score for the pair (see markdown note at the end of this notebook on how to interpret it).

In [ ]:
# @title PPI score
!./ppiscore {outfile} {N}


## 7. Generate top decoys and check restraint satisfaction

Generate the top 10 ranked complexes, then check — as a numeric sanity check independent of the `block`-based
biasing — what fraction of your 40 zein restraint residues are actually in contact with the whey protein in
each top pose (same idea as the `lgd_filter_restraints.py` step from the LightDock pipeline, reimplemented
here with Biopython). The ligand chain is relabeled `B` in each output complex so receptor and ligand can be
selected separately in PyMOL/NGLView (the raw MEGADOCK inputs both use chain A, which would otherwise collide
in the merged complex file). A CSV summary (`{LIGAND_NAME}_restraint_summary.csv`) is also written so this
ranking survives after the Colab session ends, instead of only existing as printed cell output.

In [ ]:
# @title Generate top-10 decoys + restraint contact check (+ CSV summary, chain B for ligand)
from Bio.PDB import PDBParser, NeighborSearch
import subprocess, csv

TOP_N = 10
CUTOFF = 5.0  # Angstrom, same cutoff used in the LightDock filtering step
LIGAND_CHAIN_LABEL = "B"  # relabel so receptor (A) and ligand (B) can be selected separately

def rechain(line, new_chain):
    """Rewrite the chain ID (column 22) of an ATOM/HETATM line."""
    if line.startswith("ATOM") or line.startswith("HETATM"):
        return line[:21] + new_chain + line[22:]
    return line

parser = PDBParser(QUIET=True)
pair_results = []

print(f"\n=== {LIGAND_NAME}: generating top {TOP_N} decoys ===")
for rank in range(1, TOP_N + 1):
    lig_decoy = f"{LIGAND_NAME}_lig.{rank}.pdb"
    complex_pdb = f"{LIGAND_NAME}_complex.{rank}.pdb"

    subprocess.run(["./decoygen", lig_decoy, LIGAND_PDB, outfile, str(rank)], check=True)

    with open(complex_pdb, "w") as out:
        # Receptor: keep chain A as-is
        with open(BLOCKED_RECEPTOR) as fin:
            for line in fin:
                if not line.startswith("END"):
                    out.write(line)
        # Ligand: relabel to chain B so it doesn't collide with the receptor's chain A
        with open(lig_decoy) as fin:
            for line in fin:
                if not line.startswith("END"):
                    out.write(rechain(line, LIGAND_CHAIN_LABEL))

    # Restraint contact check
    structure = parser.get_structure(complex_pdb, complex_pdb)
    atoms = list(structure.get_atoms())
    ns = NeighborSearch(atoms)

    receptor_res_in_contact = set()
    for atom in atoms:
        parent_chain = atom.get_parent().get_parent().id
        if parent_chain != chain:
            continue
        close = ns.search(atom.coord, CUTOFF)
        for other in close:
            other_chain = other.get_parent().get_parent().id
            if other_chain != chain:  # contact from the ligand side
                receptor_res_in_contact.add(atom.get_parent().id[1])
                break

    satisfied = receptor_res_in_contact & keep_residues
    fraction = len(satisfied) / len(keep_residues)
    pair_results.append((rank, len(satisfied), len(keep_residues), fraction, complex_pdb))
    print(f"  rank {rank:2d}: {len(satisfied)}/{len(keep_residues)} restraint residues in contact "
          f"({fraction:.1%})  -> {complex_pdb}")

results_sorted = sorted(pair_results, key=lambda x: -x[3])

print("\nBest pose by restraint satisfaction:")
best = results_sorted[0]
print(f"  rank {best[0]} ({best[4]}), {best[3]:.1%} restraints satisfied")

# Save the summary to CSV so it survives after this Colab session ends
summary_csv = f"{LIGAND_NAME}_restraint_summary.csv"
with open(summary_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["megadock_rank", "restraints_satisfied", "restraints_total",
                      "fraction_satisfied", "complex_pdb"])
    for rank, sat, total, frac, pdb in pair_results:  # original MEGADOCK rank order
        writer.writerow([rank, sat, total, f"{frac:.4f}", pdb])
print(f"\nWrote {summary_csv}")


## 8. Visualize the best pose

In [ ]:
# @title Enable widgets + view the best pose (by restraint satisfaction)
from google.colab import output
output.enable_custom_widget_manager()
import nglview as nv

best_complex = results_sorted[0][4]
view = nv.show_structure_file(best_complex)
view


## 9. Download results

Zips up the docking output, top complexes, and the restraint-satisfaction CSV so you can pull them back into your workspace folder.

In [ ]:
# @title Package results for download
import os, shutil
from google.colab import files

os.makedirs("results_export", exist_ok=True)
shutil.copy(outfile, "results_export/")
shutil.copy(summary_csv, "results_export/")
for rank, sat, total, frac, pdb in pair_results:
    shutil.copy(pdb, "results_export/")

zip_name = f"megadock_zein_{LIGAND_NAME}_results"
shutil.make_archive(zip_name, "zip", "results_export")
files.download(f"{zip_name}.zip")

# Also save a copy back to Drive so it persists beyond this Colab session
drive_results_dir = os.path.join(DRIVE_FOLDER, "results")
os.makedirs(drive_results_dir, exist_ok=True)
shutil.copy(f"{zip_name}.zip", drive_results_dir)
print(f"Also copied results to {drive_results_dir}/{zip_name}.zip")


## Interpreting the results

- **PPI score**: an overall interaction-likelihood estimate (Ohue et al.). Per the original tutorial's precision
  guidance, a "PPI positive" call corresponds very roughly to 10–50–80% precision depending on the score
  threshold used — treat it as a coarse screening signal, not a probability of true binding.
- **Restraint satisfaction fraction** (this notebook's addition, saved to `{LIGAND_NAME}_restraint_summary.csv`):
  what fraction of your 40 known/expected zein interaction residues are actually in contact in a given pose.
  Poses with a high fraction across *multiple* independently-ranked decoys are more trustworthy than a single
  high-scoring outlier. **Note the CSV keeps MEGADOCK's original rank order (by FFT score), not sorted by
  restraint fraction** — sort the `fraction_satisfied` column yourself if you want the restraint-best pose
  first.
- Each complex PDB now has the receptor on chain A and the ligand relabeled to chain B, so you can select
  them separately in PyMOL (`chain A` / `chain B`) instead of both landing on chain A.
- As with the LightDock/HDOCK results: none of these scores are physical binding free energies. Use them for
  relative ranking and for identifying which interface region keeps showing up, not as absolute affinity values.

## References

- Ohue M, et al. **MEGADOCK 4.0**: an ultra-high-performance protein-protein docking software for heterogeneous
  supercomputers. *Bioinformatics*, 30(22): 3281-3283, 2014. https://doi.org/10.1093/bioinformatics/btu532
- MEGADOCK GitHub: https://github.com/akiyamalab/MEGADOCK
- License: CC BY-NC 4.0 — non-commercial use only without authorization from Tokyo Institute of Technology.
